# Donut Training Diagnostics

Loads a checkpoint (or the base model) and inspects what goes in and what comes out.

In [ ]:
import sys
sys.path.insert(0, '.')

from pathlib import Path
import torch
import numpy as np
from PIL import Image
from transformers import DonutProcessor

from label_formatter import LabelFormatter
from model_module import DonutModule
from dataset import DonutDataset

# ── Config ──────────────────────────────────────────────────────────────────
MODEL_NAME   = 'naver-clova-ix/donut-base'
CKPT_PATH    = None          # e.g. 'lightning_logs/version_0/checkpoints/epoch=0-val_loss=12.3456.ckpt'
IMAGE_PATH   = '../test_data/test_data.jpg'  # single image for diagnosis
SAMPLE_FIELDS = []           # leave empty to run without a label (shows generation only)
TASK_START_TOKEN = '<s_donut>'
MAX_NEW_TOKENS = 128
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## 1. Load model + processor

In [ ]:
additional_tokens = [TASK_START_TOKEN] + LabelFormatter.get_all_tokens()

processor = DonutProcessor.from_pretrained(MODEL_NAME)
processor.tokenizer.add_special_tokens({'additional_special_tokens': additional_tokens})

if CKPT_PATH:
    module = DonutModule.load_from_checkpoint(
        CKPT_PATH,
        processor=processor,
    )
    print(f'Loaded checkpoint: {CKPT_PATH}')
else:
    module = DonutModule(
        model_name=MODEL_NAME,
        processor=processor,
        additional_tokens=additional_tokens,
    )
    print('Loaded base pretrained model (no checkpoint)')

model = module.model.to(DEVICE).eval()
tok   = processor.tokenizer

print(f'\nModule training flag : {module.training}')
print(f'model.training       : {model.training}')
print(f'encoder.training     : {model.encoder.training}')
print(f'decoder.training     : {model.decoder.training}')

## 2. Token ID sanity checks

In [ ]:
start_id  = model.config.decoder_start_token_id
pad_id    = model.config.pad_token_id
vocab_size = len(tok)

print(f'Vocab size               : {vocab_size}')
print(f'decoder_start_token_id   : {start_id}  → "{tok.decode([start_id])}"')
print(f'pad_token_id             : {pad_id}    → "{tok.decode([pad_id])}"')
print(f'Random-chance loss (nats): {np.log(vocab_size):.3f}')
print()

print('Additional token IDs:')
for t in additional_tokens:
    tid = tok.convert_tokens_to_ids(t)
    in_vocab = tid != tok.unk_token_id
    print(f'  {t:40s} → {tid}  {"OK" if in_vocab else "MISSING (maps to UNK!)"}')

## 3. Inspect pixel values from the image processor

In [ ]:
from IPython.display import display

img = Image.open(IMAGE_PATH).convert('RGB')
display(img.resize((320, 240)))
print(f'Image size: {img.size}')

pv = processor(img, return_tensors='pt').pixel_values  # (1, 3, H, W)
print(f'\npixel_values shape : {tuple(pv.shape)}')
print(f'dtype              : {pv.dtype}')
print(f'min / max / mean   : {pv.min():.4f} / {pv.max():.4f} / {pv.mean():.4f}')
print()
# Donut normalises to roughly [-1, 1] via mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5]
if pv.min() >= -1.1 and pv.max() <= 1.1:
    print('Range looks correct (roughly [-1, 1])')
else:
    print('WARNING: range is outside [-1, 1] — check processor image_size / normalization')

## 4. Label encoding / decoding round-trip

In [ ]:
# Build a fake sample — swap SAMPLE_FIELDS for real data if available
sample = {'image': IMAGE_PATH, 'fields': SAMPLE_FIELDS}
formatter = LabelFormatter()

target_text = TASK_START_TOKEN + formatter.format(sample)
print(f'Target text:\n  {target_text!r}\n')

tokenized = tok(
    target_text,
    add_special_tokens=False,
    max_length=512,
    padding='max_length',
    truncation=True,
    return_tensors='pt',
)
labels = tokenized.input_ids.squeeze(0).clone()

# Show raw token ids (non-padded only)
non_pad = labels[labels != tok.pad_token_id]
print(f'Non-pad label tokens : {len(non_pad)}')
print('Token IDs :', non_pad.tolist())
print('Decoded   :', tok.decode(non_pad))

# Verify -100 masking
labels_masked = labels.clone()
labels_masked[labels_masked == tok.pad_token_id] = -100
n_active = (labels_masked != -100).sum().item()
print(f'\nActive (non-padded) label positions: {n_active} / {len(labels)}')

## 5. Teacher-forced forward pass — what does the model predict at each position?

In [ ]:
pv_dev  = pv.to(DEVICE)
lbl_dev = labels_masked.unsqueeze(0).to(DEVICE)

with torch.no_grad():
    out = model(pixel_values=pv_dev, labels=lbl_dev)

print(f'Teacher-forced loss: {out.loss.item():.4f}')
print(f'(Random-chance baseline: {np.log(vocab_size):.3f})')
print()

# Decode argmax predictions at every active position
logits    = out.logits[0]              # (seq_len, vocab)
pred_ids  = logits.argmax(dim=-1)      # (seq_len,)
true_ids  = labels[: logits.shape[0]] # align length

active = true_ids != tok.pad_token_id
pred_active = pred_ids[active]
true_active = true_ids[active]

print('Position | True token            | Predicted token')
print('-' * 60)
for i, (t, p) in enumerate(zip(true_active.tolist(), pred_active.tolist())):
    tt = tok.decode([t])
    pt = tok.decode([p])
    match = '✓' if t == p else '✗'
    print(f'  {i:4d}   | {tt:22s} | {pt:22s} {match}')
    if i >= 30:
        print(f'  ... ({len(true_active) - 31} more positions)')
        break

## 6. Greedy auto-regressive generation (no teacher forcing)

In [ ]:
with torch.no_grad():
    gen_ids = model.generate(
        pixel_values=pv_dev,
        decoder_start_token_id=start_id,
        max_new_tokens=MAX_NEW_TOKENS,
        eos_token_id=tok.eos_token_id,
        pad_token_id=pad_id,
    )

generated_text = tok.decode(gen_ids[0], skip_special_tokens=False)
print('Generated output:')
print(repr(generated_text))
print()
print('Target text:')
print(repr(target_text))